# Build B — Threat Intel RAG Digest (Kaggle GPU run)

Prerequisites: enable GPU (Settings → Accelerator → GPU) and enable internet
access for this notebook. The repo (code + corpus) is public, so no token or
secret is needed to clone it.

To let this notebook push `digest.md`/`audit.jsonl` back to GitHub (the last
cell), you need a GitHub secret configured once:
1. On GitHub, create a fine-grained Personal Access Token scoped **only** to
   the `av4874/RAG_threatIntel` repo, with **Contents: Read and write**
   permission (no other scopes/repos).
2. In this notebook: Add-ons → Secrets → Add a new secret. Label it
   `GITHUB_TOKEN`, paste the token value, and toggle it **on** for this
   notebook.

The token is never printed or logged by the push cell below.


In [ ]:
import sys
REPO_URL = "https://github.com/av4874/RAG_threatIntel.git"

!{sys.executable} -m pip install -q transformers accelerate torch sentence-transformers faiss-cpu
!git clone {REPO_URL} /kaggle/working/repo

# Editable installs (pip install -e) write a .pth file that Python only
# reads at interpreter startup -- since this kernel is already running,
# a mid-session editable install is never picked up. Add the source
# directory to sys.path directly instead, which works immediately.
sys.path.insert(0, "/kaggle/working/repo/src")


In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)


class QwenLLMClient:
    def generate(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt"
        ).to(model.device)
        output_ids = model.generate(**inputs, max_new_tokens=512, do_sample=True, temperature=0.2)
        generated = output_ids[0][inputs["input_ids"].shape[-1]:]
        return tokenizer.decode(generated, skip_special_tokens=True)


In [ ]:
from pathlib import Path
from threat_digest.pipeline import run_pipeline

digest_path = run_pipeline(
    corpus_dir=Path("/kaggle/working/repo/corpus"),
    output_dir=Path("/kaggle/working/output"),
    llm_client=QwenLLMClient(),
    k=5,
)
print(f"Digest written to: {digest_path}")
print(digest_path.read_text())


## Validate before trusting the digest

Before treating any entry as real, open `output/audit.jsonl` and check, for a
few items: do the retrieved chunks actually support the summary and risk
score? If the model is inventing detail not present in the retrieved text,
that's a grounding failure — see the design spec's Testing/Validation section.

In [ ]:
import shutil
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

REPO_DIR = Path("/kaggle/working/repo")
OUTPUT_DIR = Path("/kaggle/working/output")
GITHUB_REPO = "av4874/RAG_threatIntel"

shutil.copy(OUTPUT_DIR / "digest.md", REPO_DIR / "digest.md")
shutil.copy(OUTPUT_DIR / "audit.jsonl", REPO_DIR / "audit.jsonl")


def run_git(args):
    result = subprocess.run(["git", *args], cwd=REPO_DIR, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"git {' '.join(args)} failed")
    return result.stdout


run_git(["config", "user.name", "kaggle-pipeline-bot"])
run_git(["config", "user.email", "kaggle-pipeline-bot@users.noreply.github.com"])
run_git(["add", "digest.md", "audit.jsonl"])

status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPO_DIR, capture_output=True, text=True
).stdout

if status.strip():
    run_git(["commit", "-m", "chore: refresh digest and audit log from Kaggle run"])

    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    push_url = f"https://{github_token}@github.com/{GITHUB_REPO}.git"

    push_result = subprocess.run(
        ["git", "push", push_url, "HEAD:master"],
        cwd=REPO_DIR, capture_output=True, text=True,
    )
    # Redact the token from any output before printing -- it can appear in
    # git's own error text on a failed push (e.g. an invalid-URL message).
    safe_stdout = push_result.stdout.replace(github_token, "***")
    safe_stderr = push_result.stderr.replace(github_token, "***")
    print(safe_stdout)
    if push_result.returncode != 0:
        print(safe_stderr)
        raise RuntimeError("git push failed")
    print("Pushed digest.md and audit.jsonl to master.")
else:
    print("No changes to commit -- digest and audit log are identical to what's already on master.")
